In [12]:
# Step 1b: download GHCN-Daily for Texas stations
import pandas as pd, duckdb

st = pd.read_fwf('data/raw/ghcnd-stations.txt', colspecs=[(0,11),(12,20),(21,30),(38,68)], names=['STATION','LAT','LON','NAME'])
con = duckdb.connect()

# Đọc toàn bộ các file CSV thực tế có sẵn
df = con.execute("""SELECT * FROM read_csv_auto('data/by_station/*.csv', union_by_name=true)""").df()

# Chuẩn hóa tên cột ID thành STATION
df = df.rename(columns={'ID': 'STATION'})

# Lấy các trạm thực tế có trong thư mục CSV kết hợp thông tin tọa độ từ file stations.txt
available_stations = df['STATION'].unique()
picked = st[st['STATION'].isin(available_stations)].head(100)

# Lọc DataFrame theo các trạm hợp lệ thực tế
df = df[df.STATION.isin(picked.STATION)]
df['DATE'] = pd.to_datetime(df['DATE'].astype(str), errors='coerce')

print(df.shape, df.STATION.nunique())

(289823, 8) 100


In [13]:
from ydata_profiling import ProfileReport
import os

# Tạo thư mục report nếu chưa có
os.makedirs('../report', exist_ok=True)

# Lấy mẫu tối đa 20.000 dòng để profiling chạy nhanh hơn
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('../report/profile.html')
df.describe(include='all').T.to_csv('../report/table_describe_raw.csv')

print('Đã tạo xong profile.html và table_describe_raw.csv thành công!')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_30532\882159412.py:1: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<00:00, 555.08it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Đã tạo xong profile.html và table_describe_raw.csv thành công!


In [ ]:
# Step 2: pivot elements to columns, clean quality flags, convert units safely(Chuyển đổi các phần tử thời tiết thành cột, làm sạch cờ chất lượng, chuyển đổi đơn vị)
df = df[df.Q_FLAG.isna()]                        # drop values with a quality flag
w = df.pivot_table(index=['STATION', 'DATE'], columns='ELEMENT', values='DATA_VALUE').reset_index()

# Chỉ định chuẩn hóa chia 10 cho các cột thực sự có mặt trong dữ liệu
existing_elements = [col for col in ['TMAX', 'TMIN', 'PRCP', 'TAVG'] if col in w.columns]
if existing_elements:
    w[existing_elements] = w[existing_elements] / 10

if 'TMAX' in w.columns:
    w = w[(w.TMAX > -40) & (w.TMAX < 60)]

# Resample và nội suy theo từng trạm
numeric_cols = [col for col in existing_elements if col in ['TMAX', 'TMIN', 'PRCP']]
w = (w.set_index('DATE').groupby('STATION')[numeric_cols].resample('D').mean().reset_index())

if 'TMAX' in w.columns:
    w[['TMAX']] = w.groupby('STATION')[['TMAX']].transform(lambda x: x.interpolate(limit=3))
    ok = w.groupby('STATION').TMAX.apply(lambda x: x.notna().mean()) >= 0.95
    w = w[w.STATION.isin(ok[ok].index)]

import os
os.makedirs('data/processed', exist_ok=True)
w.to_parquet('data/processed/ghcn_tx.parquet', index=False)
print(w.shape, w.STATION.nunique())

(249195, 3) 100


In [17]:
# Re-run the cleaning steps as functions safely checking available columns
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])

# Chỉ định các bước làm sạch dựa trên cột thực tế có trong w
cleaning_steps = [
    ('drop duplicates', lambda x: x.drop_duplicates(['STATION', 'DATE']), 'same unit and timestamp'),
]
if 'TMAX' in w.columns:
    cleaning_steps.append(('drop missing target', lambda x: x.dropna(subset=['TMAX']), 'cannot be forecast'))

_, cleaning_log = clean_log(w, cleaning_steps)
import os
os.makedirs('report', exist_ok=True)
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False)
print(cleaning_log)

              step  rows_before  rows_after  dropped                   reason
0  drop duplicates       249195      249195        0  same unit and timestamp
